# abfv MVP - Step 1: generate the Fv 3D structure

Implements README item 2: *"Generate a 3D structure from the sequences, e.g. with ABodyBuilder3."*

Predicts the paired VH<->VL **Fv** structure with the ABodyBuilder3 base model and writes a PDB
(`out/complex.pdb`, chains **A**=heavy / **B**=light).

Paths and sequences are hardcoded for now - they'll move behind the Rust CLI later. 
Subsequent steps (split chains -> FreeSASA -> ΔSASA contacts -> visualize) will be added below.

In [4]:
import os
import torch
from abodybuilder3.utils import string_to_input, output_to_pdb, add_atom37_to_output
from abodybuilder3.lightning_module import LitABB3

CKPT = "/home/filip/CloudStation/Python/abodybuilder3/output/plddt-loss/best_second_stage.ckpt"
OUT_DIR = "/home/filip/CloudStation/Python/abfv/out"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

device: cuda


In [5]:
# Antibody Fv input sequences (heavy = VH, light = VL)
heavy = "EVQLVESGGGLVQPGGSLRLSCAASGYTFTNYGMNWVRQAPGKGLEWVGWINTYTGEPTYAADFKRRFTFSLDTSKSTAYLQMNSLRAEDTAVYYCAKYPHYYGSSHWYFDVWGQGTLVTVSS"
light = "DIQMTQSPSSLSASVGDRVTITCSASQDISNYLNWYQQKPGKAPKVLIYFTSSLHSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYSTVPWTFGQGTKVEIK"

print(f"VH (heavy): {len(heavy)} residues")
print(f"VL (light): {len(light)} residues")

VH (heavy): 123 residues
VL (light): 107 residues


In [6]:
module = LitABB3.load_from_checkpoint(CKPT, map_location=device)
model = module.model.to(device).eval()

print("ABodyBuilder3 base model loaded")

ABodyBuilder3 base model loaded


In [8]:
# Predict the Fv structure: sequences -> PDB
ab_input = string_to_input(heavy=heavy, light=light)

batch = {k: (v.unsqueeze(0).to(device) if k not in ["single", "pair"] else v.to(device))
         for k, v in ab_input.items()}

with torch.no_grad():
    output = model(batch, batch["aatype"])

output = add_atom37_to_output(output, ab_input["aatype"].to(device))
pdb = output_to_pdb(output, ab_input)

out_path = os.path.join(OUT_DIR, "complex.pdb")
with open(out_path, "w") as fh:
    fh.write(pdb)

# sanity check: chains present and residues per chain
chains = {}
for line in pdb.splitlines():
    if line.startswith("ATOM"):
        chains.setdefault(line[21], set()).add(line[22:26])
        
print(f"wrote {out_path}")
print("residues per chain:", {c: len(rs) for c, rs in sorted(chains.items())})

IndentationError: expected an indented block (3735543559.py, line 23)